In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

In [ ]:
try:
    result = json.load(open("choice_result.json", "r"))
except:
    result = {}
BENCHMARK_TASKS = {}
for clas_task in pmlb.classification_dataset_names:
    BENCHMARK_TASKS[clas_task] = {
        "dataset": clas_task,
        "task": "classification",
        "metric": "accuracy",
        "size": "m",
    }
for reg_task in pmlb.regression_dataset_names:
    BENCHMARK_TASKS[reg_task] = {
        "dataset": reg_task,
        "task": "regression",
        "metric": "r2",
        "size": "m",
    }
ITERS_SIZE = {
    "xl": 4,
    "l": 8,
    "m": 12,
    "s": 20,
}

In [ ]:
BENCHMARK_TASKS

In [ ]:
df_summary = pmlb.dataset_lists.df_summary
df_summary

In [ ]:
from benchmark.pmlb.runbenchmark import _download_failure

def runner_flaml(task, use_spark=False):
    conf = BENCHMARK_TASKS[task]
    try:
        X, y = pmlb.fetch_data(conf['dataset'], return_X_y=True)
        print(f"Running on dataset: {conf['dataset']}")
    except (OSError, ValueError) as e:
        return _download_failure(conf, e)
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25)

    aml = flaml.AutoML()

    automl_settings = {
        "max_iter": ITERS_SIZE.get(conf["size"], 4),
        "metric": conf["metric"],
        "task": conf["task"],
        "estimator_list": ['lgbm', 'rf', 'xgboost', 'extra_tree', 'xgb_limitdepth'],
    }
    
    if use_spark:
        automl_settings["use_spark"] = True
        automl_settings["n_concurrent_trials"] = 2
    
    if conf["task"] == "classification":
        automl_settings["estimator_list"] += ["lrl1"]
        
    start = time.time()
    aml.fit(X_train=X_train, y_train=y_train, **automl_settings)
    end = time.time()
    
    pred = aml.predict(X_test)
    
    if conf["task"] == "classification":
        score = accuracy_score(y_test, pred)
    else:
        score = r2_score(y_test, pred)
    del X, y, X_train, y_train, X_test, y_test, pred, aml
    return {
        "score": score,
        "duration": end - start,
        "metric": conf["metric"],
        "status": "ok",
        "error": None,
    }


In [ ]:
for task in BENCHMARK_TASKS.keys():
    if task not in result.keys():
        result[task] = runner_flaml(task)
        json.dump(result, open("result.json", "w"))
        gc.collect()

In [ ]:
selected_tasks = []
for task in result.keys():
    metrics = result[task]
    if metrics.get("status", "ok") != "ok" or "metric" not in metrics or metrics.get("score") is None:
        continue
    metric = result[task]["metric"]
    score = result[task]["score"]
    # Retain datasets where FLAML performed well, yet still possess room for improvement
    if 0.9 >= score >= 0.6:
        print(f"Task {task} score of {metric}: {score}")
        selected_tasks.append({"dataset":task,"metric":metric, "result":score})

In [ ]:
datasets_n_dict = dict(zip(df_summary["dataset"].values.tolist(), df_summary["n_instances"].values.tolist()))

In [ ]:
for i in range(len(selected_tasks)):
   selected_tasks[i]["n"] = datasets_n_dict[selected_tasks[i]["dataset"]]
selected_tasks.sort(key=lambda x: x["n"])

selected_cla_tasks = [task for task in selected_tasks if task["metric"] == "accuracy"]
selected_reg_tasks = [task for task in selected_tasks if task["metric"] == "r2"]

In [ ]:
import numpy as np
select_n = 7
def choice_task(tasks):
    selected_i = np.geomspace(1, len(tasks), select_n).astype('int')
    selected_i = len(tasks) - selected_i
    choice = []
    for ii in range(len(selected_i)):
        i = selected_i[ii]
        new_task = {}
        new_task["dataset"] = tasks[i]["dataset"]
        new_task["task"] = "regression" if tasks[i]["metric"] == "r2" else "classification"
        new_task["metric"] = tasks[i]["metric"]
        if i == len(tasks) - 1:
            size = "xl"
        elif i == 0:
            size = "s"
        elif ii <= len(selected_i) / 2:
            size = "l"
        else:
            size = "m"
        new_task["size"] = size
        choice.append(new_task)
    
    return choice

choice_cla_tasks = choice_task(selected_cla_tasks)
choice_reg_tasks = choice_task(selected_reg_tasks)
final_choice_tasks = choice_reg_tasks + choice_cla_tasks

In [ ]:
final_choice_tasks

In [ ]:
json.dump(final_choice_tasks, open("choice_tasks.json", "w"))